# import libs

In [1]:
import nbimporter
%run -i Common_Functions.ipynb
from Common_Functions import *

In [32]:
# Custom normalization (no zeros)
def normalize_no_zero(series, epsilon=0.):
    min_val = series.min()
    max_val = series.max()
    range_val = max_val - min_val
    return epsilon + (1 - epsilon) * (series - min_val) / range_val

# process data

## power generation

In [2]:
file_path = "NEM Registration and Exemption List.xlsx"
df_participants = pd.read_excel(file_path, sheet_name='PU and Scheduled Loads')

In [3]:
df_scada_hour = pd.read_csv('processed_data_files/SCADA_NOTJOINED_FILTER_SCADAgreaterthan0_groupby_hour_duid_sum.csv')
df_participants_select = df_participants[['Fuel Source - Descriptor', 'DUID', 'Technology Type - Primary', 'Region']]
df_scada_merged_participants = df_scada_hour.merge(df_participants_select, on="DUID", how="left")
df_scada_merged_participants = df_scada_merged_participants[df_scada_merged_participants['DATE'] < '2025-01-01']

In [4]:
df_scada_merged_participants.head()

,DATE,HOUR,DUID,SCADAVALUE,Fuel Source - Descriptor,Technology Type - Primary,Region
0,2019-01-01,0,AGLHAL,0.00,Natural Gas / Diesel,Combustion,SA1
1,2019-01-01,0,AGLSOM,0.00,Natural Gas,Combustion,VIC1
2,2019-01-01,0,ANGAST1,0.00,Diesel,Combustion,SA1
3,2019-01-01,0,ARWF1,500.60,Wind,Renewable,VIC1
4,2019-01-01,0,BALBG1,4.07,NaN,NaN,NaN


In [9]:
df_scada_merged_participants_qld = df_scada_merged_participants[df_scada_merged_participants['Region']=='QLD1']

In [11]:
df_scada_merged_participants_qld.count()

DATE                         4559338
HOUR                         4559338
DUID                         4559338
SCADAVALUE                   4559338
Fuel Source - Descriptor     4559338
Technology Type - Primary    4559338
Region                       4559338
dtype: int64

In [13]:
df_scada_merged_participants_qld.dropna(subset = ['Fuel Source - Descriptor', 'Technology Type - Primary'], inplace=True)

/var/folders/b6/bmslj24d11n41826_8m7yb400000gn/T/ipykernel_1563/3717290291.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_scada_merged_participants_qld.dropna(subset = ['Fuel Source - Descriptor', 'Technology Type - Primary'], inplace=True)


In [14]:
df_scada_merged_participants_qld.count()

DATE                         4559338
HOUR                         4559338
DUID                         4559338
SCADAVALUE                   4559338
Fuel Source - Descriptor     4559338
Technology Type - Primary    4559338
Region                       4559338
dtype: int64

In [15]:
df_scada_merged_participants_qld.head()

,DATE,HOUR,DUID,SCADAVALUE,Fuel Source - Descriptor,Technology Type - Primary,Region
8,2019-01-01,0,BARCALDN,0.00,Natural Gas,Combustion,QLD1
9,2019-01-01,0,BARCSF1,5.50,Solar,Renewable,QLD1
10,2019-01-01,0,BARRON-1,366.84,Water,Renewable,QLD1
11,2019-01-01,0,BARRON-2,364.80,Water,Renewable,QLD1
24,2019-01-01,0,BRAEMAR1,0.00,Coal Seam Methane,Combustion,QLD1


In [19]:
df_scada_merged_participants_qld['DATE'] = pd.to_datetime(df_scada_merged_participants_qld['DATE'])
df_scada_merged_participants_qld['Month'] = df_scada_merged_participants_qld['DATE'].dt.month
df_scada_merged_participants_qld['Year'] = df_scada_merged_participants_qld['DATE'].dt.year
df_scada_merged_participants_qld['DayName'] = df_scada_merged_participants_qld['DATE'].dt.day_name()
df_scada_merged_participants_qld['YearMonth'] = df_scada_merged_participants_qld['DATE'].dt.to_period('M')

/var/folders/b6/bmslj24d11n41826_8m7yb400000gn/T/ipykernel_1563/3882317625.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_scada_merged_participants_qld['DATE'] = pd.to_datetime(df_scada_merged_participants_qld['DATE'])
/var/folders/b6/bmslj24d11n41826_8m7yb400000gn/T/ipykernel_1563/3882317625.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_scada_merged_participants_qld['Month'] = df_scada_merged_participants_qld['DATE'].dt.month
/var/folders/b6/bmslj24d11n41826_8m7yb400000gn/T/ipykernel_1563/

In [20]:
df_scada_merged_participants_qld.head()

,DATE,HOUR,DUID,SCADAVALUE,Fuel Source - Descriptor,Technology Type - Primary,Region,Month,Year,DayName,YearMonth
8,2019-01-01,0,BARCALDN,0.00,Natural Gas,Combustion,QLD1,1,2019,Tuesday,2019-01
9,2019-01-01,0,BARCSF1,5.50,Solar,Renewable,QLD1,1,2019,Tuesday,2019-01
10,2019-01-01,0,BARRON-1,366.84,Water,Renewable,QLD1,1,2019,Tuesday,2019-01
11,2019-01-01,0,BARRON-2,364.80,Water,Renewable,QLD1,1,2019,Tuesday,2019-01
24,2019-01-01,0,BRAEMAR1,0.00,Coal Seam Methane,Combustion,QLD1,1,2019,Tuesday,2019-01


In [33]:
df_scada_merged_participants_qld['SCADAVALUE_norm'] = normalize_no_zero(df_scada_merged_participants_qld['SCADAVALUE'])

/var/folders/b6/bmslj24d11n41826_8m7yb400000gn/T/ipykernel_1563/4115636203.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_scada_merged_participants_qld['SCADAVALUE_norm'] = normalize_no_zero(df_scada_merged_participants_qld['SCADAVALUE'])


In [34]:
df_scada_merged_participants_qld.to_csv("processed_data_for_dashboard/PowerGeneration_QLD.csv", index=False)

## pricing and demand

In [17]:
df_price = pd.read_csv('processed_data_files/PRICE_groupby_hour_regionid_avg.csv')
df_demand = pd.read_csv('processed_data_files/DEMAND_groupby_hour_regionid_sum.csv')

In [18]:
df_price.head(3)

,REGIONID,Date_Hour,RRP
0,NSW1,2019-01-01 00:00:00,68.191979
1,NSW1,2019-01-01 01:00:00,71.477868
2,NSW1,2019-01-01 02:00:00,65.966705


In [23]:
df_price_qld = df_price[df_price['REGIONID'] == 'QLD1']

In [24]:
df_price_qld['Date_Hour'] = pd.to_datetime(df_price_qld['Date_Hour'])
df_price_qld['Month'] = df_price_qld['Date_Hour'].dt.month
df_price_qld['Year'] = df_price_qld['Date_Hour'].dt.year
df_price_qld['DayName'] = df_price_qld['Date_Hour'].dt.day_name()
df_price_qld['YearMonth'] = df_price_qld['Date_Hour'].dt.to_period('M')
df_price_qld['Hour'] = df_price_qld['Date_Hour'].dt.hour

/var/folders/b6/bmslj24d11n41826_8m7yb400000gn/T/ipykernel_1563/1904534803.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_price_qld['Date_Hour'] = pd.to_datetime(df_price_qld['Date_Hour'])
/var/folders/b6/bmslj24d11n41826_8m7yb400000gn/T/ipykernel_1563/1904534803.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_price_qld['Month'] = df_price_qld['Date_Hour'].dt.month
/var/folders/b6/bmslj24d11n41826_8m7yb400000gn/T/ipykernel_1563/1904534803.py:3: SettingWithCopyWarning: 
A value is trying to be

In [25]:
df_price_qld.head()

,REGIONID,Date_Hour,RRP,Month,Year,DayName,YearMonth,Hour
53353,QLD1,2019-01-01 00:00:00,64.605948,1,2019,Tuesday,2019-01,0
53354,QLD1,2019-01-01 01:00:00,66.544864,1,2019,Tuesday,2019-01,1
53355,QLD1,2019-01-01 02:00:00,61.140174,1,2019,Tuesday,2019-01,2
53356,QLD1,2019-01-01 03:00:00,52.883149,1,2019,Tuesday,2019-01,3
53357,QLD1,2019-01-01 04:00:00,49.505579,1,2019,Tuesday,2019-01,4


In [26]:
df_price_qld.count()

REGIONID     53353
Date_Hour    53353
RRP          53353
Month        53353
Year         53353
DayName      53353
YearMonth    53353
Hour         53353
dtype: int64

In [35]:
df_price_qld['RRP_norm'] = normalize_no_zero(df_price_qld['RRP'])

/var/folders/b6/bmslj24d11n41826_8m7yb400000gn/T/ipykernel_1563/1873789629.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_price_qld['RRP_norm'] = normalize_no_zero(df_price_qld['RRP'])


In [36]:
df_price_qld.to_csv("processed_data_for_dashboard/ElectricityPrice_QLD.csv", index=False)

In [29]:
df_demand_qld = df_demand[df_demand['REGIONID'] == 'QLD1']

In [30]:
df_demand_qld['Date_Hour'] = pd.to_datetime(df_demand_qld['Date_Hour'])
df_demand_qld['Month'] = df_demand_qld['Date_Hour'].dt.month
df_demand_qld['Year'] = df_demand_qld['Date_Hour'].dt.year
df_demand_qld['DayName'] = df_demand_qld['Date_Hour'].dt.day_name()
df_demand_qld['YearMonth'] = df_demand_qld['Date_Hour'].dt.to_period('M')

/var/folders/b6/bmslj24d11n41826_8m7yb400000gn/T/ipykernel_1563/987686436.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_demand_qld['Date_Hour'] = pd.to_datetime(df_demand_qld['Date_Hour'])
/var/folders/b6/bmslj24d11n41826_8m7yb400000gn/T/ipykernel_1563/987686436.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_demand_qld['Month'] = df_demand_qld['Date_Hour'].dt.month
/var/folders/b6/bmslj24d11n41826_8m7yb400000gn/T/ipykernel_1563/987686436.py:3: SettingWithCopyWarning: 
A value is trying to b

In [31]:
df_demand_qld.head()

,REGIONID,Date_Hour,RESDEMAND,Month,Year,DayName,YearMonth
2223,QLD1,2019-01-01,115674373.0,1,2019,Tuesday,2019-01
2224,QLD1,2019-01-02,123473992.0,1,2019,Wednesday,2019-01
2225,QLD1,2019-01-03,122851746.0,1,2019,Thursday,2019-01
2226,QLD1,2019-01-04,122740848.0,1,2019,Friday,2019-01
2227,QLD1,2019-01-05,118949941.0,1,2019,Saturday,2019-01


In [37]:
df_demand_qld['RESDEMAND_norm'] = normalize_no_zero(df_demand_qld['RESDEMAND'])

/var/folders/b6/bmslj24d11n41826_8m7yb400000gn/T/ipykernel_1563/2521860667.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_demand_qld['RESDEMAND_norm'] = normalize_no_zero(df_demand_qld['RESDEMAND'])


In [38]:
df_demand_qld.head()

,REGIONID,Date_Hour,RESDEMAND,Month,Year,DayName,YearMonth,RESDEMAND_norm
2223,QLD1,2019-01-01,115674373.0,1,2019,Tuesday,2019-01,0.660278
2224,QLD1,2019-01-02,123473992.0,1,2019,Wednesday,2019-01,0.743625
2225,QLD1,2019-01-03,122851746.0,1,2019,Thursday,2019-01,0.736975
2226,QLD1,2019-01-04,122740848.0,1,2019,Friday,2019-01,0.735790
2227,QLD1,2019-01-05,118949941.0,1,2019,Saturday,2019-01,0.695280


In [39]:
df_demand_qld.to_csv("processed_data_for_dashboard/DEMAND_QLD.csv", index=False)